In [11]:
# pip install alpaca-py pandas
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
import pandas as pd

API_KEY = "YOUR_KEY"
SECRET_KEY = "YOUR_SECRET"

client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

def fetch_1m_last_week(symbols):
    req = StockBarsRequest(
        symbol_or_symbols=symbols,     # ["AAPL","MSFT"] veya "AAPL,MSFT"
        timeframe=TimeFrame.Minute,
        start=pd.Timestamp.utcnow() - pd.Timedelta(days=7),
        end=pd.Timestamp.utcnow(),
        limit=10000
    )
    bars = client.get_stock_bars(req).df  # MultiIndex: symbol, timestamp
    # kolonlar genelde: open, high, low, close, volume, trade_count, vwap ...
    return bars.reset_index()

df = fetch_1m_last_week(["AAPL","MSFT","NVDA"])
print(df.head())

APIError: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx</center>
</body>
</html>


In [56]:
from twelvedata import TDClient
import pandas as pd
from datetime import datetime, timedelta, timezone

# 🔐 BURAYA KENDI YENI API KEY'INI YAZ
API_KEY = "fec0f77b5bb346caa53531b642b58878"

td = TDClient(apikey=API_KEY)


def fetch_1m_last_week(symbols):
    """
    symbols: str veya list
    return: pandas DataFrame (multi-index: symbol, datetime)
    """

    if isinstance(symbols, str):
        symbols = [symbols]

    end = datetime.now(timezone.utc)
    start = end - timedelta(days=7)

    all_data = []

    for symbol in symbols:
        try:
            ts = td.time_series(
                symbol=symbol,
                interval="1min",
                outputsize=5000,   # mümkün olan maksimum
                timezone="UTC"
            ).as_pandas()

            ts.index = pd.to_datetime(ts.index, utc=True)
            ts = ts.sort_index()

            # son 1 hafta filtre
            ts = ts[(ts.index >= start) & (ts.index <= end)]

            ts["symbol"] = symbol
            all_data.append(ts)

        except Exception as e:
            print(f"Hata ({symbol}):", e)

    if not all_data:
        return pd.DataFrame()

    df = pd.concat(all_data)
    df = df.reset_index().rename(columns={"datetime": "timestamp"})
    # df = df.set_index(["symbol", "timestamp"]).sort_index()

    return df


# 🔥 ÖRNEK KULLANIM
symbols = ["ULKER.IS"]
df = fetch_1m_last_week(symbols)


Hata (ULKER.IS): **symbol** or **figi** parameter is missing or invalid. Please provide a valid symbol according to API documentation: https://twelvedata.com/docs#reference-data


In [66]:
from twelvedata import TDClient
import pandas as pd
from datetime import datetime, timedelta, timezone

API_KEY = "fec0f77b5bb346caa53531b642b58878"
td = TDClient(apikey=API_KEY)

def fetch_bist_1m(symbol):
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=7)

    # .IS zaten varsa tekrar ekleme
    if not symbol.endswith(".IS"):
        symbol = f"{symbol}.IS"

    ts = td.time_series(
        symbol=symbol,
        interval="1min",
        outputsize=5000,
        timezone="UTC"
    ).as_pandas()

    ts.index = pd.to_datetime(ts.index, utc=True)
    ts = ts.sort_index()
    ts = ts[(ts.index >= start) & (ts.index <= end)]

    return ts

df = fetch_bist_1m("ULKER")
print(df.head())

TwelveDataError: **symbol** or **figi** parameter is missing or invalid. Please provide a valid symbol according to API documentation: https://twelvedata.com/docs#reference-data

In [72]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import time

API_KEY = "BURAYA_YENI_KEY"

def fetch_bist_1m_multi(symbols, batch_size=20, sleep_sec=1.0):
    """
    symbols: ["ULKER", "GARAN", ...] veya ["ULKER:XIST", ...]
    return: DataFrame (MultiIndex: symbol, datetime) with OHLCV strings
    """

    # XIST ekle (yoksa)
    fixed = []
    for s in symbols:
        s = s.strip().upper()
        if ":" not in s:
            s = f"{s}:XIST"
        fixed.append(s)

    end = datetime.now(timezone.utc)
    start = end - timedelta(days=7)

    all_frames = []

    for i in range(0, len(fixed), batch_size):
        batch = fixed[i:i+batch_size]
        symbol_param = ",".join(batch)

        url = "https://api.twelvedata.com/time_series"
        params = {
            "symbol": symbol_param,
            "interval": "1min",
            "outputsize": 5000,
            "apikey": API_KEY,
            "format": "JSON",
            # istersen timezone paramı da ekleyebilirsin:
            # "timezone": "UTC",
        }

        r = requests.get(url, params=params, timeout=60)
        payload = r.json()

        # Batch cevap genelde { "SYM1": {...}, "SYM2": {...} } şeklinde gelir.
        # Ama bazen tek sembolde {status, values} döner. İkisini de handle edelim.
        if "values" in payload and "meta" in payload:
            payload = {payload["meta"]["symbol"]: payload}

        for sym, obj in payload.items():
            if not isinstance(obj, dict):
                continue

            if obj.get("status") == "error":
                # sembol bazlı hata
                # ör: invalid symbol / permission / credits
                print(f"[SKIP] {sym}: {obj.get('message')}")
                continue

            values = obj.get("values")
            if not values:
                print(f"[EMPTY] {sym}: values boş")
                continue

            df = pd.DataFrame(values)
            df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
            df = df.set_index("datetime").sort_index()
            df = df[(df.index >= start) & (df.index <= end)]
            df["symbol"] = sym
            all_frames.append(df)

        time.sleep(sleep_sec)

    if not all_frames:
        return pd.DataFrame()

    out = pd.concat(all_frames, axis=0)
    out = out.reset_index().set_index(["symbol", "datetime"]).sort_index()
    return out

# örnek:
symbols = ["ULKER", "GARAN", "THYAO", "SISE"]
df = fetch_bist_1m_multi(symbols, batch_size=10, sleep_sec=1.2)
print(df.head())
print(df.index.get_level_values(0).unique()[:10])

Empty DataFrame
Columns: []
Index: []
RangeIndex(start=0, stop=0, step=1)


In [73]:
df

""


In [80]:
from ib_insync import IB, Stock, util
import pandas as pd

ib = IB()
ib.connect("127.0.0.1", 7497, clientId=1)  # 7497=paper, 7496=live (çoğunlukla)

# BIST için exchange genelde 'IBIS2' (TWS'ta contract details ile doğrula)
# currency TRY
contract = Stock("ULKER", "IBIS2", "TRY")

bars = ib.reqHistoricalData(
    contract,
    endDateTime="",
    durationStr="1 W",
    barSizeSetting="1 min",
    whatToShow="TRADES",
    useRTH=True,
    formatDate=1
)

df = util.df(bars)  # columns: date, open, high, low, close, volume, ...
print(df.head())

ib.disconnect()

RuntimeError: This event loop is already running

API connection failed: ConnectionRefusedError(61, "Connect call failed ('127.0.0.1', 7497)")
Make sure API port on TWS/IBG is open


In [78]:
pip install ib_insync

  Using cached ib_insync-0.9.86-py3-none-any.whl.metadata (5.4 kB)
  Using cached eventkit-1.0.3-py3-none-any.whl.metadata (5.4 kB)
Using cached ib_insync-0.9.86-py3-none-any.whl (72 kB)
Using cached eventkit-1.0.3-py3-none-any.whl (31 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ib_insync]
Note: you may need to restart the kernel to use updated packages.


In [92]:
import yfinance as yf
import pandas as pd
import time
import random

def fetch_1m_7d(ticker, retries=6, base_sleep=2.0):
    """
    ticker: 'ULKER.IS'
    return: DataFrame (DatetimeIndex) veya boş DF
    """
    last_err = None
    for attempt in range(retries):
        try:
            t = yf.Ticker(ticker)
            df = t.history(period="7d", interval="1m", auto_adjust=False, prepost=False)

            # bazen kolon isimleri farklı olabilir; normalize edelim
            if df is not None and not df.empty:
                df = df.rename(columns={
                    "Open": "open", "High": "high", "Low": "low",
                    "Close": "close", "Volume": "volume"
                })
                return df

            return pd.DataFrame()

        except Exception as e:
            last_err = e
            # exponential backoff + jitter
            sleep_s = base_sleep * (2 ** attempt) + random.uniform(0, 1.0)
            time.sleep(sleep_s)

    print(f"[FAIL] {ticker}: {last_err}")
    return pd.DataFrame()

def fetch_many(tickers):
    out = {}
    for tk in tickers:
        df = fetch_1m_7d(tk)
        if not df.empty:
            out[tk] = df
        else:
            print(f"[EMPTY] {tk}")
        # küçük bir jitter: Yahoo throttling’i azaltır
        time.sleep(random.uniform(0.2, 0.8))
    return out

tickers = [""]
data = fetch_many(tickers)

# birleştir (MultiIndex kolonlar)
if data:
    panel = pd.concat(data, axis=1)  # columns: (ticker, field)
    print(panel.head())
else:
    print("Hepsi boş döndü.")

AAPL: No price data found, symbol may be delisted (period=7d)


[EMPTY] AAPL
Hepsi boş döndü.


In [94]:
df

Empty DataFrame
Columns: [(ULKER.IS, Open), (ULKER.IS, High), (ULKER.IS, Low), (ULKER.IS, Close), (ULKER.IS, Adj Close), (ULKER.IS, Volume), (CIMSA.IS, Open), (CIMSA.IS, High), (CIMSA.IS, Low), (CIMSA.IS, Close), (CIMSA.IS, Adj Close), (CIMSA.IS, Volume), (DOAS.IS, Open), (DOAS.IS, High), (DOAS.IS, Low), (DOAS.IS, Close), (DOAS.IS, Adj Close), (DOAS.IS, Volume)]
Index: []

In [101]:
t = yf.Ticker('ULKER.IS')
df = t.history(period="7d", interval="1m", auto_adjust=False, prepost=False)

ULKER.IS: No price data found, symbol may be delisted (period=7d)


In [113]:
from defeatbeta_api.data.ticker import Ticker
ticker = Ticker("AAPL")

In [ ]:
pip uninstall defeatbeta_api

Found existing installation: defeatbeta-api 0.0.44
Uninstalling defeatbeta-api-0.0.44:
  Would remove:
    /Users/irfanakgul/anaconda3/lib/python3.11/site-packages/defeatbeta_api-0.0.44.dist-info/*
    /Users/irfanakgul/anaconda3/lib/python3.11/site-packages/defeatbeta_api/*
Proceed (Y/n)? 

In [5]:
import requests
import time

API_KEY = "d60bqkhr01qgk0vil07gd60bqkhr01qgk0vil080"
symbol = "AAPL"

# Zamanı hesapla (Unix timestamp)
to_time = int(time.time())
from_time = to_time - 7 * 24 * 60 * 60

url = f"https://finnhub.io/api/v1/stock/candle?symbol=AAPL&resolution=1&from=1700000000&to=1700600000&token=API_KEY"

response = requests.get(url)
data = response.json()

print(data)

{'error': 'Invalid API key.'}


In [11]:
import requests
import time

API_KEY = "d6eraj9r01qvn4o0otg0d6eraj9r01qvn4o0otgg"
symbol = "AAPL"

# Son 7 gün
to_time = int(time.time())
from_time = to_time - 7 * 24 * 60 * 60

url = (
    f"https://finnhub.io/api/v1/stock/candle"
    f"?symbol={symbol}"
    f"&resolution=1"
    f"&from={from_time}"
    f"&to={to_time}"
    f"&token={API_KEY}"
)

response = requests.get(url)
data = response.json()

print(data)

{'error': "You don't have access to this resource."}


In [119]:
# pip install finnhub-python
# GUNCEL VERI DE VERIYOR AMA SADECE USA VOLUME YOK
import finnhub

client = finnhub.Client(api_key="d6eraj9r01qvn4o0otg0d6eraj9r01qvn4o0otgg")

res = client.quote("AAPL")
print(res)
# print(client.symbol_lookup("ULKER"))

{'c': 272.14, 'd': 5.96, 'dp': 2.2391, 'h': 274.89, 'l': 267.71, 'o': 267.86, 'pc': 266.18, 't': 1771966800}


In [105]:
import pandas as pd
pd.DataFrame([client.symbol_lookup("ASELS")['result']][0])

,description,displaySymbol,symbol,type
0,Aselsan Elektronik Sanayi ve Ticaret AS,ASELS.E.IS,ASELS.E.IS,Common Stock


In [145]:
# DAKIKALIK VERI VERIYPR
from massive import RESTClient

client = RESTClient("ONE3QQU7xWsv0J_oC8HMmRrL6rbRmMw7")

aggs = []
for a in client.list_aggs(
    "AAPL",
    1,
    "minute",
    "2026-01-01",
    "2026-02-24",
    limit=50000,
):
    aggs.append(a)

In [246]:
from yahooquery import Ticker

ticker = Ticker("AAPL")

df = ticker.history(
    period="7d",     # son 7 gün
    interval="1m",start='2026-02-17').reset_index()

,symbol,date,open,high,low,close,volume
0,AAPL,2026-02-17 09:30:00-05:00,258.049988,258.179993,255.552994,256.440002,1654509
1,AAPL,2026-02-17 09:31:00-05:00,256.454987,258.000000,256.179993,257.980011,388654
2,AAPL,2026-02-17 09:32:00-05:00,257.899994,258.380005,257.760101,258.084991,264373
3,AAPL,2026-02-17 09:33:00-05:00,258.100006,259.000000,257.609985,258.790009,183561
4,AAPL,2026-02-17 09:34:00-05:00,258.760010,259.489990,258.700012,258.700012,262007
...,...,...,...,...,...,...,...
2336,AAPL,2026-02-24 15:56:00-05:00,272.290009,272.380005,272.260010,272.299988,120846
2337,AAPL,2026-02-24 15:57:00-05:00,272.309998,272.429901,272.230011,272.279999,180207
2338,AAPL,2026-02-24 15:58:00-05:00,272.279999,272.359985,272.220001,272.239990,227013
2339,AAPL,2026-02-24 15:59:00-05:00,272.239990,272.329987,272.067810,272.170013,11510291
